In [0]:
%sql
-- Reset the table for a clean implementation
DROP TABLE IF EXISTS silver.dim_customers;

CREATE TABLE silver.dim_customers (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- Surrogate Key
    customer_id INT,                                  -- Natural Key
    name STRING,
    address STRING,
    is_current BOOLEAN,
    start_date DATE,
    end_date DATE
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
# Initial landing data for Day 1
initial_customers = [
    (1, "Shiwam Kumar Suman", "Delhi"),
    (2, "Anuranjan Singh", "Gurgaon"),
    (3, "Akash Sharma", "Bangalore"),
    (4, "Satya Mishra", "Kolkata")
]

columns = ["customer_id", "name", "address"]

# Write to Bronze as the starting point
spark.createDataFrame(initial_customers, columns) \
     .write.mode("overwrite") \
     .saveAsTable("bronze.raw_customers")

print("Initial Bronze table created with 4 records.")

In [0]:
from pyspark.sql import functions as F

# Define the High Date constant
HIGH_DATE = '9999-12-31'

# 1. Load latest landing data from Bronze
# 2. Add metadata for the 'Current' version
new_customers_df = spark.table("bronze.raw_customers") \
    .withColumn("start_date", F.current_date()) \
    .withColumn("is_current", F.lit(True)) \
    .withColumn("end_date", F.to_date(F.lit(HIGH_DATE)))

# 3. Create a Temp View for the SQL Merge
new_customers_df.createOrReplaceTempView("source_view")

In [0]:
spark.sql(f"""
MERGE INTO silver.dim_customers AS target
USING (
  -- Part A: Current records to be inserted
  SELECT source_view.customer_id as merge_key, source_view.* FROM source_view
  UNION ALL
  -- Part B: Null keys to trigger updates on existing active records that changed
  SELECT NULL as merge_key, source_view.* FROM source_view
  JOIN silver.dim_customers ON source_view.customer_id = silver.dim_customers.customer_id
  WHERE silver.dim_customers.is_current = true 
    AND source_view.address <> silver.dim_customers.address
) AS source
ON target.customer_id = source.merge_key
WHEN MATCHED AND target.is_current = true AND target.address <> source.address THEN
  -- Logic for the OLD record: Expire it
  UPDATE SET target.is_current = false, target.end_date = source.start_date
WHEN NOT MATCHED THEN
  -- Logic for the NEW record: Insert it with the High Date
  INSERT (customer_id, name, address, is_current, start_date, end_date)
  VALUES (source.customer_id, source.name, source.address, true, source.start_date, source.end_date)
""")

In [0]:
%sql
select * from silver.dim_customers

In [0]:
# --- ROUND 1: Initial Load ---
data_r1 = [(1, "Shiwam", "Delhi")]
spark.createDataFrame(data_r1, ["customer_id", "name", "address"]) \
     .write.mode("overwrite").saveAsTable("bronze.raw_customers")

In [0]:
spark.sql(f"""
MERGE INTO silver.dim_customers AS target
USING (
  -- Part A: Current records to be inserted
  SELECT source_view.customer_id as merge_key, source_view.* FROM source_view
  UNION ALL
  -- Part B: Null keys to trigger updates on existing active records that changed
  SELECT NULL as merge_key, source_view.* FROM source_view
  JOIN silver.dim_customers ON source_view.customer_id = silver.dim_customers.customer_id
  WHERE silver.dim_customers.is_current = true 
    AND source_view.address <> silver.dim_customers.address
) AS source
ON target.customer_id = source.merge_key
WHEN MATCHED AND target.is_current = true AND target.address <> source.address THEN
  -- Logic for the OLD record: Expire it
  UPDATE SET target.is_current = false, target.end_date = source.start_date
WHEN NOT MATCHED THEN
  -- Logic for the NEW record: Insert it with the High Date
  INSERT (customer_id, name, address, is_current, start_date, end_date)
  VALUES (source.customer_id, source.name, source.address, true, source.start_date, source.end_date)
""")

In [0]:
%sql
select * from silver.dim_customers

In [0]:
# --- ROUND 2: The Change (Move to Bengaluru) ---
data_r2 = [(1, "Shiwam", "Bengaluru")] 
spark.createDataFrame(data_r2, ["customer_id", "name", "address"]) \
     .write.mode("overwrite").saveAsTable("bronze.raw_customers")

In [0]:
spark.sql(f"""
MERGE INTO silver.dim_customers AS target
USING (
  -- Part A: Current records to be inserted
  SELECT source_view.customer_id as merge_key, source_view.* FROM source_view
  UNION ALL
  -- Part B: Null keys to trigger updates on existing active records that changed
  SELECT NULL as merge_key, source_view.* FROM source_view
  JOIN silver.dim_customers ON source_view.customer_id = silver.dim_customers.customer_id
  WHERE silver.dim_customers.is_current = true 
    AND source_view.address <> silver.dim_customers.address
) AS source
ON target.customer_id = source.merge_key
WHEN MATCHED AND target.is_current = true AND target.address <> source.address THEN
  -- Logic for the OLD record: Expire it
  UPDATE SET target.is_current = false, target.end_date = source.start_date
WHEN NOT MATCHED THEN
  -- Logic for the NEW record: Insert it with the High Date
  INSERT (customer_id, name, address, is_current, start_date, end_date)
  VALUES (source.customer_id, source.name, source.address, true, source.start_date, source.end_date)
""")

In [0]:
%sql
select * from silver.dim_customers ORDER BY customer_id, end_date

In [0]:
%sql
select * from silver.fact_transactions limit 2

In [0]:
%sql
select * from silver.dim_customers limit 2